# TP Final — Entrenamiento y seguimiento con MLflow
Dataset UCI Heart Failure Clinical Records. Comparación justa de regresión logística, Random Forest y LightGBM. `time` se excluye para evitar fuga de información del seguimiento.

In [0]:
%pip install lightgbm scikit-learn mlflow pandas matplotlib seaborn

In [0]:
import os, json, tempfile, uuid
from sklearn.base import clone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             precision_score, recall_score, accuracy_score,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay)
from lightgbm import LGBMClassifier

In [0]:
# Leer archivo CSV guardado en la etapa 1
ruta = '/Volumes/workspace/bigdata_drrach/landing/tp_final_heart_failure/heart_failure.csv'
df = pd.read_csv(ruta)
assert len(df) == 299, f'Se esperaban 299 pacientes; se leyeron {len(df)}'
assert 'DEATH_EVENT' in df.columns
assert set(df.DEATH_EVENT.dropna().unique()) <= {0, 1}
assert not df.DEATH_EVENT.isna().any()
X = df.drop(columns=['DEATH_EVENT', 'time'], errors='ignore')
y = df['DEATH_EVENT'].astype(int)
assert X.shape[1] == 11, f'Se esperaban 11 predictores basales, hay {X.shape[1]}: {X.columns.tolist()}'
assert X.select_dtypes(exclude='number').empty, 'Revisar variables no numéricas'
print('Datos:', df.shape, '| Predictores:', X.shape[1])
print('Variables:', X.columns.tolist())
print('Distribución de clases:', y.value_counts().to_dict())

Datos: (299, 13) | Predictores: 11
Variables: ['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex', 'smoking']
Distribución de clases: {0: 203, 1: 96}


In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('Train:', len(X_train), 'Test:', len(X_test))

Train: 239 Test: 60


In [0]:
modelos = {
    'Regresion_logistica': Pipeline([
        ('imputacion', SimpleImputer(strategy='median')),
        ('escalado', StandardScaler()),
        ('modelo', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
    ]),
    'Random_Forest': Pipeline([
        ('imputacion', SimpleImputer(strategy='median')),
        ('modelo', RandomForestClassifier(n_estimators=200, min_samples_leaf=3,
                                         class_weight='balanced', random_state=42, n_jobs=-1))
    ]),
    'LightGBM': Pipeline([
        ('imputacion', SimpleImputer(strategy='median')),
        ('modelo', LGBMClassifier(n_estimators=100, learning_rate=0.05,
                                 num_leaves=7, min_child_samples=15,
                                 verbosity=-1, random_state=42, n_jobs=1))
    ])
}

# Cinco configuraciones iniciales y barrido de C con tres valores.
base = modelos
modelos = {
 'Regresion_logistica': clone(base['Regresion_logistica']),
 'Random_Forest': clone(base['Random_Forest']),
 'LightGBM': clone(base['LightGBM']),
 'RF_depth3': clone(base['Random_Forest']).set_params(modelo__max_depth=3),
 'LGBM_50_arboles': clone(base['LightGBM']).set_params(modelo__n_estimators=50),
}
for c in [0.01, 0.1, 10.0]:
    modelos[f'LR_barrido_C_{c}'] = clone(base['Regresion_logistica']).set_params(modelo__C=c)
session_id = uuid.uuid4().hex
print('Sesión:', session_id, '| Configuraciones:', len(modelos))

Sesión: 18d5a6f44f7547bd8b5681a185721f66 | Configuraciones: 8


In [0]:
# Databricks: experiment en el workspace; ruta única por usuario/carpeta.
# Si no hay permisos para crear el experimento, cambiá esta ruta por una ruta válida en tu workspace.
usuario = spark.sql('SELECT current_user() AS usuario').first()['usuario']
experimento = f'/Users/{usuario}/TP_Final_Insuficiencia_Cardiaca'
mlflow.set_experiment(experimento)
print('Experimento:', experimento)

Experimento: /Users/racheluja@gmail.com/TP_Final_Insuficiencia_Cardiaca


In [0]:
resultados = []
for nombre, pipeline in modelos.items():
    with mlflow.start_run(run_name=nombre):
        cv_scores = cross_validate(
            pipeline, X_train, y_train, cv=cv,
            scoring={'roc_auc': 'roc_auc', 'f1': 'f1', 'recall': 'recall'},
            n_jobs=1, error_score='raise'
        )
        cv_auc = float(np.mean(cv_scores['test_roc_auc']))
        cv_std = float(np.std(cv_scores['test_roc_auc']))
        pipeline.fit(X_train, y_train)
        probas = pipeline.predict_proba(X_test)[:, 1]
        pred = (probas >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
        metricas = {
            'cv_roc_auc_mean': cv_auc,
            'cv_roc_auc_std': cv_std,
            'cv_f1_mean': float(np.mean(cv_scores['test_f1'])),
            'cv_recall_mean': float(np.mean(cv_scores['test_recall'])),
            'test_roc_auc': float(roc_auc_score(y_test, probas)),
            'test_average_precision': float(average_precision_score(y_test, probas)),
            'test_f1': float(f1_score(y_test, pred, zero_division=0)),
            'test_precision': float(precision_score(y_test, pred, zero_division=0)),
            'test_sensitivity': float(recall_score(y_test, pred, zero_division=0)),
            'test_specificity': float(tn / (tn + fp)),
            'test_accuracy': float(accuracy_score(y_test, pred)),
        }
        mlflow.log_params({'seed': 42, 'test_size': 0.20, 'cv_folds': 5, 'session_id': session_id,
                           'threshold': 0.5, 'excluded_followup_time': True,
                           'n_train': len(X_train), 'n_test': len(X_test)})
        mlflow.log_params({k: str(v) for k, v in pipeline.get_params(deep=True).items()
                           if k.startswith('modelo__')})
        mlflow.log_metrics(metricas)
        mlflow.set_tags({'dataset': 'UCI Heart Failure Clinical Records (id=519)',
                         'target': 'DEATH_EVENT', 'model_family': type(pipeline.named_steps['modelo']).__name__,
                         'session_id': session_id, 'study': 'heart_failure_extended_v1',
                         'note': 'Exploratory educational study, not clinical deployment'})
        with tempfile.TemporaryDirectory() as tmp:
            fig, ax = plt.subplots(figsize=(5, 4))
            ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=[0,1], ax=ax)
            ax.set_title(f'Matriz de confusión — {nombre}')
            fig.tight_layout()
            fig.savefig(os.path.join(tmp, 'matriz_confusion.png'), dpi=150)
            plt.close(fig)
            fig, ax = plt.subplots(figsize=(5, 4))
            RocCurveDisplay.from_predictions(y_test, probas, ax=ax)
            ax.set_title(f'Curva ROC — {nombre}')
            fig.tight_layout()
            fig.savefig(os.path.join(tmp, 'curva_roc.png'), dpi=150)
            plt.close(fig)
            pd.DataFrame({'y_real': y_test.to_numpy(), 'probabilidad': probas,
                          'prediccion': pred}).to_csv(os.path.join(tmp, 'predicciones_test.csv'), index=False)
            mlflow.log_artifacts(tmp, artifact_path='evaluacion')
        mlflow.log_dict({'features': X.columns.tolist(), 'train_indices': X_train.index.tolist(),
                         'test_indices': X_test.index.tolist(), 'session_id': session_id}, 'particion.json')
        firma = infer_signature(X_train, pipeline.predict(X_train))
        mlflow.sklearn.log_model(pipeline, artifact_path='modelo', signature=firma,
                                 input_example=X_train.head(3))
        resultados.append({'modelo': nombre, **metricas})
        print(nombre, '| CV AUC:', round(cv_auc, 3), '| Test AUC:', round(metricas['test_roc_auc'], 3))

Regresion_logistica | CV AUC: 0.804 | Test AUC: 0.734


Random_Forest | CV AUC: 0.779 | Test AUC: 0.791


LightGBM | CV AUC: 0.754 | Test AUC: 0.77


RF_depth3 | CV AUC: 0.779 | Test AUC: 0.789


LGBM_50_arboles | CV AUC: 0.748 | Test AUC: 0.801


LR_barrido_C_0.01 | CV AUC: 0.79 | Test AUC: 0.763


LR_barrido_C_0.1 | CV AUC: 0.801 | Test AUC: 0.75


LR_barrido_C_10.0 | CV AUC: 0.804 | Test AUC: 0.734


In [0]:
comparacion = pd.DataFrame(resultados).sort_values('cv_roc_auc_mean', ascending=False)
display(comparacion)
print('Modelo seleccionado según CV ROC-AUC:', comparacion.iloc[0]['modelo'])
print('Nota: el conjunto test NO se usa para seleccionar el modelo.')
# Para entregar, exportar el notebook con resultados y capturar la pantalla de MLflow Experiments.

modelo,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_recall_mean,test_roc_auc,test_average_precision,test_f1,test_precision,test_sensitivity,test_specificity,test_accuracy
LR_barrido_C_10.0,0.8044444444444444,0.04806972842600283,0.6758906033633801,0.7658333333333334,0.7342747111681642,0.5913748957259026,0.5128205128205128,0.5,0.5263157894736842,0.7560975609756098,0.6833333333333333
Regresion_logistica,0.804068023989899,0.04953993125322106,0.6686833961561729,0.7533333333333333,0.7342747111681642,0.5913748957259026,0.5263157894736842,0.5263157894736842,0.5263157894736842,0.7804878048780488,0.7
LR_barrido_C_0.1,0.800954861111111,0.05166237802723042,0.6695300751879699,0.7541666666666667,0.7496790757381258,0.6156367977424069,0.5263157894736842,0.5263157894736842,0.5263157894736842,0.7804878048780488,0.7
LR_barrido_C_0.01,0.7901799242424243,0.05501123099554926,0.6301156791479372,0.6766666666666666,0.7625160462130937,0.6370360079291293,0.5405405405405406,0.5555555555555556,0.5263157894736842,0.8048780487804879,0.7166666666666667
Random_Forest,0.778962279040404,0.046924211437415236,0.5602224388514712,0.5433333333333332,0.7907573812580231,0.5868162743201591,0.5555555555555556,0.5882352941176471,0.5263157894736842,0.8292682926829268,0.7333333333333333
RF_depth3,0.7785700757575758,0.043417641107451506,0.5922154720819904,0.595,0.7894736842105263,0.6553638876819219,0.5789473684210527,0.5789473684210527,0.5789473684210527,0.8048780487804879,0.7333333333333333
LightGBM,0.7542747790404041,0.029644618315668315,0.5615341085929321,0.5058333333333334,0.7702182284980744,0.5648236358222914,0.5142857142857142,0.5625,0.47368421052631576,0.8292682926829268,0.7166666666666667
LGBM_50_arboles,0.7475686553030303,0.040264374244186885,0.5515752395752396,0.49416666666666664,0.8010269576379974,0.5939817310258613,0.5161290322580645,0.6666666666666666,0.42105263157894735,0.9024390243902439,0.75


Modelo seleccionado según CV ROC-AUC: LR_barrido_C_10.0
Nota: el conjunto test NO se usa para seleccionar el modelo.
